In [31]:
import json
import pandas as pd

# Đọc dữ liệu từ file general_result.json
with open("../test_gemma2_2b_all_techniques_2026_04_13/general_result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for waf, waf_data in data.items():
    for attack_type, attack_data in waf_data.items():
        for phase, phase_data in attack_data.items():
            total_payload = phase_data.get("total_payload", 0)
            bypassed = phase_data.get("bypassed", {}).get("total_payload", 0)
            blocked = phase_data.get("blocked", {}).get("total_payload", 0)
            row = {
                "WAF": waf,
                "Attack Type": attack_type,
                "Phase": phase,
                "Total Payload": total_payload,
                "BYPASSED": bypassed,
                "bypassed safe": 0,
                "bypassed harm": 0,
                "BLOCKED": blocked,
                "blocked safe": 0,
                "blocked harm": 0,
            }
            # Thêm các trường harmness nếu có
            for status in ["bypassed", "blocked"]:
                harmness = phase_data.get(status, {}).get("harmness", {})
                if "safe" in harmness:
                    percent = float(harmness["safe"]) * 100 / total_payload
                    row[f"{status} safe"] = f"{harmness["safe"]} ({percent:.0f}%)"
                if "harm" in harmness:
                    percent = float(harmness["harm"]) * 100 / total_payload
                    row[f"{status} harm"] = f"{harmness["harm"]} ({percent:.0f}%)"
            rows.append(row)
            row["BYPASSED"] = f"{bypassed} ({(bypassed / total_payload * 100 if total_payload else 0):.0f}%)"
            row["BLOCKED"] = f"{blocked} ({(blocked / total_payload * 100 if total_payload else 0):.0f}%)"

# Tạo DataFrame và hiển thị
pd.set_option('display.max_columns', None)
df = pd.DataFrame(rows)
df.head(40)  # Hiển thị 40 dòng đầu tiên

,WAF,Attack Type,Phase,Total Payload,BYPASSED,bypassed safe,bypassed harm,BLOCKED,blocked safe,blocked harm
0,ModSecurity,xss_dom,PHASE_1,50,9 (18%),4 (8%),5 (10%),41 (82%),6 (12%),35 (70%)
1,ModSecurity,xss_dom,PHASE_3,50,35 (70%),17 (34%),18 (36%),15 (30%),10 (20%),5 (10%)
2,ModSecurity,xss_reflected,PHASE_1,50,7 (14%),4 (8%),3 (6%),43 (86%),9 (18%),34 (68%)
3,ModSecurity,xss_reflected,PHASE_3,50,42 (84%),16 (32%),26 (52%),8 (16%),3 (6%),5 (10%)
4,ModSecurity,xss_stored,PHASE_1,50,38 (76%),7 (14%),31 (62%),12 (24%),0 (0%),12 (24%)
5,ModSecurity,xss_stored,PHASE_3,50,44 (88%),17 (34%),27 (54%),6 (12%),6 (12%),0 (0%)
6,ModSecurity,sql_injection,PHASE_1,50,18 (36%),11 (22%),7 (14%),32 (64%),27 (54%),5 (10%)
7,ModSecurity,sql_injection,PHASE_3,50,45 (90%),26 (52%),19 (38%),5 (10%),5 (10%),0 (0%)
8,ModSecurity,sql_injection_blind,PHASE_1,50,24 (48%),7 (14%),17 (34%),26 (52%),20 (40%),6 (12%)
9,ModSecurity,sql_injection_blind,PHASE_3,50,42 (84%),16 (32%),26 (52%),8 (16%),8 (16%),0 (0%)
